
## MarkdownTextSplitter（基于长度与 Markdown 分隔符分割）

MarkdownTextSplitter 是 RecursiveCharacterTextSplitter 的子类。它会在 Markdown 的特定语法元素（如标题、代码块、水平线等）处进行分割，并结合 chunk_size（块大小）和 chunk_overlap（重叠大小）来控制文本块的长度

## MarkdownHeaderTextSplitter（按 Markdown 标题层级结构分割）

如果你希望严格按照 Markdown 的标题层级（如 #、##、###）来拆分文档，并将对应的标题作为元数据（Metadata）保留在每一个分片中，推荐使用 MarkdownHeaderTextSplitter。这在 RAG（检索增强生成）中非常有用，能帮你保留上下文层级。

使用 MarkdownHeaderTextSplitter 时，只需指定目标标题层级及对应的元数据字段名称即可进行拆分，分割后的文本块会自动携带层级 Metadata。完整代码与进阶用法可参考 LangChain中文网。

## 两者如何选择？

• MarkdownHeaderTextSplitter：适合结构清晰的长文档（如技术文档、API说明书），按标题切分并保留层级 Metadata，适合做上下文检索。
• MarkdownTextSplitter：适合单标题下内容过多、或需严格限制字符/Token 数量的场景，在 Markdown 语法与长度限制间折中。
### 进阶技巧：在实际 RAG 生产中常组合使用——先用 MarkdownHeaderTextSplitter 按标题粗切并保留元数据，再用 RecursiveCharacterTextSplitter 进行精细化二次切分。

### 举例1

In [1]:
from langchain_text_splitters import MarkdownTextSplitter

# 初始化 MarkdownTextSplitter
markdown_text = """
# 第一章：介绍
这是第一章的介绍内容。

## 1.1 背景
这里是背景部分的详细描述。
- 点 A
- 点 B

```python
# 这是一段代码块
def hello():
    print("Hello, world!")
```

---

# 第二章：进阶
这是第二章的内容。
"""

# 设置每块的最大字符数和重叠字符数
splitter = MarkdownTextSplitter(chunk_size=100, chunk_overlap=20)

# 分割文本
docs = splitter.create_documents([markdown_text])

for i, doc in enumerate(docs):
    print(f"--- 文本块 {i+1} ---")
    print(doc.page_content)
    print(f"Metadata: {doc.metadata}")

--- 文本块 1 ---
# 第一章：介绍
这是第一章的介绍内容。

## 1.1 背景
这里是背景部分的详细描述。
- 点 A
- 点 B
Metadata: {}
--- 文本块 2 ---
```python
# 这是一段代码块
def hello():
    print("Hello, world!")
Metadata: {}
--- 文本块 3 ---
```

---

# 第二章：进阶
这是第二章的内容。
Metadata: {}


### 举例2

In [2]:
from langchain_text_splitters import MarkdownTextSplitter

markdown_text = """
# 一级标题\n
这是一级标题下的内容\n\n
## 二级标题\n
- 二级下列表项1\n
- 二级下列表项2\n
"""

# 关键步骤：直接修改实例属性
splitter = MarkdownTextSplitter(chunk_size=30, chunk_overlap=0)
splitter._is_separator_regex = True  #  强制将分隔符视为正则表达式

# 执行分割
docs = splitter.create_documents(texts = [markdown_text])

# print(len(docs))

for i, doc in enumerate(docs):
    print(f"\n🔍 分块 {i + 1}:")
    print(doc.page_content)


🔍 分块 1:
# 一级标题

这是一级标题下的内容

🔍 分块 2:
## 二级标题

- 二级下列表项1

- 二级下列表项2
